# 01 — Exploração e obtenção dos dados (Rota Livre)

Baixa as bases públicas, converte tudo para YOLO com as 3 classes do Rota Livre e baixa fotos brasileiras do Mapillary para anotação.

**Ambiente:** Colab gratuito (CPU basta neste notebook).

In [ ]:
# === Configuração: rode sempre que abrir o notebook ===
REPO_URL = "https://github.com/Lucas-Gomes-Alcantara/Rota-Livre-Visao.git"  # <- troque pelo repositório da equipe
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/rota_livre"          # pasta persistente (o disco do Colab é apagado ao desconectar)
os.makedirs(DRIVE, exist_ok=True)
if not os.path.exists("/content/Rota-Livre-Visao"):
    !git clone -q $REPO_URL /content/Rota-Livre-Visao
%cd /content/Rota-Livre-Visao
!pip install -q -r requirements.txt

In [ ]:
# Chaves guardadas em "Secrets" do Colab (ícone de chave na barra lateral) — nunca escreva a chave no notebook.
for k in ("ROBOFLOW_API_KEY", "MAPILLARY_TOKEN"):
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print(f"{k} não configurada (necessária só para a etapa correspondente)")

## 1. Bases do Roboflow Universe
Potholes (ODbL), Sidewalk Walkability (CC BY 4.0), Sidewalk Obstacles (CC BY 4.0). A versão usada fica registrada em `data/raw/<base>/SOURCE.txt`.

In [ ]:
!python src/download_data.py roboflow
!cat data/raw/*/SOURCE.txt

## 2. (Opcional) RDD2022
Só ligue (`enabled: true` em `configs/sources.yaml`) se a classe *buraco* ficar com poucos exemplos. Primeiro liste arquivos, tamanhos e licença:

In [ ]:
!python src/download_data.py rdd2022 --list
# !python src/download_data.py rdd2022 --pattern Japan

## 3. Conversão para YOLO + remapeamento de classes

In [ ]:
!python src/convert.py

import json
rep = json.load(open('results/metrics/convert_report.json'))
for fonte, r in rep.items():
    print(f"\n== {fonte}")
    print('  classes originais :', r['caixas_por_classe_original'])
    print('  classes Rota Livre:', r['caixas_por_classe_rota_livre'])
    print('  estatísticas      :', r['estatisticas'])

In [ ]:
import pandas as pd
m = pd.read_csv('data/interim/manifest.csv')
cols = [c for c in m.columns if c.startswith('n_')]
display(m.groupby('source')[cols].sum())
display(m.groupby('source')[['width','height']].describe().round(0))
print('grupos por fonte:'); print(m.groupby('source')['group'].nunique())

## 4. Fotos brasileiras do Mapillary (CC BY-SA 4.0) para anotar
Token gratuito em mapillary.com/dashboard/developers → salve como `MAPILLARY_TOKEN` nos Secrets.
Depois: suba o zip no Roboflow (projeto gratuito), anote seguindo `docs/annotation_guide.md` e exporte em YOLO.

In [ ]:
!python src/mapillary_fetch.py --city sao_paulo rio_de_janeiro belo_horizonte --n 300 --require-sidewalk
!cd data/raw && zip -qr $DRIVE/mapillary_br_para_anotar.zip mapillary_br
print('zip salvo no Drive:', DRIVE)

## 5. Salvar progresso no Drive

In [ ]:
!zip -qr $DRIVE/interim.zip data/interim results/metrics
print('ok')